<a href="https://colab.research.google.com/github/Developer3009/Codesoft-Internship/blob/Genre-Classification/Genre_Classification_Dataset_IMDb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Define a text cleaning function
def clean_text(text):
    # Remove special characters, numbers, and make lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower()
    return text

# 2. Load the datasets
# The dataset uses ' ::: ' as a separator. We use engine='python' to handle multi-character separators.
columns = ['ID', 'TITLE', 'GENRE', 'DESCRIPTION']
#Genre Classification Dataset/
#Genre Classification Dataset/
print("Loading data...")
train_df = pd.read_csv('train_data.txt', sep=' ::: ', engine='python', names=columns)
test_df = pd.read_csv('test_data_solution.txt', sep=' ::: ', engine='python', names=columns)

# 3. Apply cleaning to the plot descriptions
print("Cleaning text data...")
train_df['CLEAN_DESC'] = train_df['DESCRIPTION'].apply(clean_text)
test_df['CLEAN_DESC'] = test_df['DESCRIPTION'].apply(clean_text)

# 4. Separate Features (X) and Target (y)
X_train = train_df['CLEAN_DESC']
y_train = train_df['GENRE']

X_test = test_df['CLEAN_DESC']
y_test = test_df['GENRE']

print(f"Training samples: {len(X_train)} | Testing samples: {len(X_test)}")

In [ ]:
# 1. Initialize TF-IDF Vectorizer
# stop_words='english' automatically removes common words like 'the', 'is', 'and'
# max_features limits the vocabulary to the top 50,000 most important words to manage memory
print("Vectorizing text data...")
tfidf = TfidfVectorizer(stop_words='english', max_features=50000)

# Fit on training data and transform both train and test sets
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# 2. Initialize and Train the Classifier
print("Training Logistic Regression model...")
# max_iter is increased to ensure the solver converges on high-dimensional text data
model = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
model.fit(X_train_tfidf, y_train)

print("Model trained successfully.")

In [ ]:
# 1. Generate Predictions
print("Generating predictions on test set...")
y_pred = model.predict(X_test_tfidf)

# 2. Evaluate Performance
accuracy = accuracy_score(y_test, y_pred)
print(f"\nOverall Accuracy: {accuracy:.4f}\n")

print("Classification Report:")
# zero_division=0 prevents warnings if some rare genres are never predicted
print(classification_report(y_test, y_pred, zero_division=0))

# 3. Test the model with a custom plot summary
def predict_genre(plot_summary):
    cleaned = clean_text(plot_summary)
    vectorized = tfidf.transform([cleaned])
    prediction = model.predict(vectorized)[0]
    return prediction

# Feel free to change this text to test the model!
sample_plot = "A group of astronauts travel through a wormhole in search of a new habitable planet for humanity as Earth faces environmental collapse."
predicted = predict_genre(sample_plot)
print(f"\nSample Plot: {sample_plot}")
print(f"Predicted Genre: {predicted}")